In [ ]:
import os
import pandas as pd
import commons as c

# Merge and save DFs for equiv and normal

In [ ]:
def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Loop through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path)
            # Append the DataFrame to the list
            dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df

# Function to split the name column and create new columns
def split_name_column(name):
    name = name.replace('.qasm', '')
    parts = name.split('_')
    position = int(parts[5].replace('P', ''))
    qubit = parts[6].replace('Q', '')
    
    if len(parts) > 7: 
        parameters = parts[7].strip('[]') 
    else: 
        parameters = None

    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters

def get_gate_type(gate):
    single_qubit_gates = ["x", "h", "p", "t", "s", "z", "y", "id", "rx", "ry", "rz", "sx", "u", "u1", "u2", "u3"]
    multi_qubit_gates = ["swap", "rzz", "rxx", "cx", "cz", "cp", "ccx", "cswap", "ch"]
    if gate in single_qubit_gates:
        return 'Single-qubit'
    elif gate in multi_qubit_gates:
        return 'Multi-qubit'
    else:
        return 'Gate not supported'
    
# Function to categorize position based on percentage
def categorize_position(percentage):
    if percentage <= 20:
        return 'Beginning'
    elif percentage <= 40:
        return 'Pre middle'
    elif percentage <= 60:
        return 'Middle'
    elif percentage <= 80:
        return 'Post middle'
    else:
        return 'End'

In [ ]:
def process_characteristics(file_path):
    """Processes the characteristics Excel file into a DataFrame."""
    df_charac = pd.read_excel(file_path, usecols=[0, 2, 3, 5, 6, 7])
    df_charac['algo'] = df_charac.iloc[:, 0].str.split('_').str[0]
    df_charac['qubits'] = df_charac['qubits'].astype(str)
    return df_charac.drop(columns=[df_charac.columns[0]])

In [ ]:
def get_dataframe(model, mutant, df_char):
    """Generates a processed DataFrame for a given noise model, mutant type, and threshold."""

    # Get all the results in a df
    folder_path = f'./results/distances/{model}_{mutant}'
    df = read_and_merge_csv_files(folder_path)
    
    # Categorize input type
    df['Input_type'] = df['Input'].str.split('_').str[0]

    # Split 'Name' column into multiple columns
    df[['Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))

    # Categorize gate type
    df['Gate_type'] = df['Gate'].apply(get_gate_type)

    # Calculate position percentage and categorize
    df['max_position'] = df.groupby(['Algorithm', 'Qubits_number'])['Position'].transform('max')
    df['position_percentage'] = (df['Position'] / df['max_position']) * 100
    df['Relative_position'] = df['position_percentage'].apply(categorize_position)

    # Drop intermediate columns
    # df = df.drop(columns=['max_position', 'position_percentage', 'Name'])

    # Merge with characteristics DataFrame
    merged_df = pd.merge(df_char, df, left_on=['qubits', 'algo'], right_on=['Qubits_number', 'Algorithm'], how='right')
    merged_df = merged_df.drop(columns=['qubits', 'algo'])

    # Map output types
    merged_df['Output_type'] = merged_df['Algorithm'].map(c.output_type)

    # Save DataFrame to CSV
    csv_path = f'results/dataframes/{model}_{mutant}.csv'
    merged_df.to_csv(csv_path, mode='w', header=True, index=False)

    return merged_df

In [ ]:
xlsx_path = 'data/origin_qc/programs_characteristics.xlsx'
df_charac = process_characteristics(xlsx_path)
os.makedirs('results/dataframes/', exist_ok=True)

mutant_typed_df = {"equiv": [], "normal": []}

for hw in c.hardware:
    # Load 'equiv' and 'normal' DataFrames once
    df_equiv = get_dataframe(hw, "equiv", df_charac)
    df_normal = get_dataframe(hw, "normal", df_charac)
    df_equiv['nature'] = "Equivalent mutant"
    df_normal['nature'] = "Non-Equivalent mutant"
        
    mutant_temp_df = {"equiv": df_equiv, "normal": df_normal}
        
    for mutant_type, df in mutant_temp_df.items():
        df['hardware'] = hw
        mutant_typed_df[mutant_type].append(df)

### Adding Thresholds

In [ ]:
def process_metrics(complete_df):
    """Processes metrics and saves results to CSV."""

    selected_columns = complete_df[[
        'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Input', 'Input_type',
        'Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits',
        'Gate_type', 'Relative_position', 'Output_type', 'hardware', 'nature' 
    ]]

    new_rows = []

    for metric, metric_name in c.metrics.items():
        for threshold in c.thresholds:
            metric_df = selected_columns.copy()
            metric_df['metric'] = metric
            metric_df['metric_full'] = metric_df['metric'].map(c.metric_names)
            metric_df['threshold'] = threshold
            metric_df['ideal_distance'] = complete_df[f'Ideal_{metric_name}']
            metric_df['noisy_distance'] = complete_df[f'Noisy_{metric_name}']
            
            tolerance_series = metric_df['hardware'].apply(lambda hw: c.get_tolerance_values(threshold, model=hw)[metric_name])
            if metric == 'F':
                metric_df['ideal_label'] = complete_df[f'Ideal_{metric_name}'] < c.get_tolerance_values('I')[metric_name]
                metric_df['noisy_label'] = complete_df[f'Noisy_{metric_name}'] < tolerance_series
            else:
                metric_df['ideal_label'] = complete_df[f'Ideal_{metric_name}'] > c.get_tolerance_values('I')[metric_name]
                metric_df['noisy_label'] = complete_df[f'Noisy_{metric_name}'] > tolerance_series

            metric_df['correctness'] = metric_df['ideal_label'] == metric_df['noisy_label']
            metric_df['hardware_named'] = metric_df['hardware'].map(c.hardware_names)
            new_rows.append(metric_df)

    final_df = pd.concat(new_rows, ignore_index=True)
    final_df = final_df.astype(c.type_dict)
    return final_df

In [ ]:
df_equiv_0 = pd.concat(mutant_typed_df.get("equiv"), ignore_index=True)
df_normal_0 = pd.concat(mutant_typed_df.get("normal"), ignore_index=True)
df_all_mutants = pd.concat([df_equiv_0, df_normal_0], ignore_index=True)
new_df = process_metrics(df_all_mutants)
output_folder = 'results/dataframes/'
os.makedirs(output_folder, exist_ok=True)
output_path = os.path.join(output_folder, f'results_all_mutants.csv')
new_df.to_csv(output_path, index=False)

In [ ]:
new_df.columns